# YOLO26 live object detection

Run the setup cell once, then run the camera cell. Press **q** in the camera window to stop.

In [1]:
%pip install -U ultralytics opencv-python

  Using cached opencv_python-5.0.0.93-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.14.0.94-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

camera = cv2.VideoCapture(0)
blur_faces = False

try:
    while True:
        ok, frame = camera.read()
        if not ok:
            break

        if blur_faces:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            faces = face_cascade.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(30, 30)
            )

            for x, y, w, h in faces:
                face = frame[y:y+h, x:x+w]

                # Ensure the blur kernel is large enough and odd-sized
                frame[y:y+h, x:x+w] = cv2.GaussianBlur(
                    face,
                    (51, 51),
                    0
                )

        result = model(frame, conf=0.35, verbose=False)[0]
        annotated_frame = result.plot()

        cv2.imshow("YOLO26 camera detection", annotated_frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break
        elif key in (ord("b"), ord("B")):
            blur_faces = not blur_faces
            print("Face blur:", "ON" if blur_faces else "OFF")

finally:
    camera.release()
    cv2.destroyAllWindows()

Face blur: ON
Face blur: OFF
Face blur: ON
Face blur: OFF
